# NB_04 — GDT Applicability to TES Fabrication v1.0

**Engineering question**

> Can the General Divisor Theorem directly specify admissible TES-fabrication states from the constraints currently represented in `sensors-becker`?

This notebook is an applicability audit, not a forced application.

For

\[
S(N,L,m,a)=\{1\le n\le L:n\equiv a\pmod m,\ \gcd(n,N)=1\},
\]

write

\[
d=\operatorname{rad}(\gcd(m,N)),\qquad
R=\frac{\operatorname{rad}(N)}{d}.
\]

If \(\gcd(a,d)=1\), the exact minimal period is

\[
T_{\min}=mR=\operatorname{lcm}(m,\operatorname{rad}(N)),
\]

with \(\varphi(R)\) accepted values per period. If \(\gcd(a,d)>1\), the accepted set is empty.

A direct TES application therefore requires a physically meaningful integer state, modulus, residue constraint, and coprimality constraint. Arbitrary numerical binning does not count.


## Workflow

```text
SOURCE_01–SOURCE_04
        +
Engineering Objects
        ↓
TES fabrication variables
        ↓
GDT hypothesis audit
        ├── hypotheses supported → exact admissible states
        └── hypotheses absent    → explicit blockers / missing specification
```


## 1. Locate repository and load current sensor state


In [ ]:
from __future__ import annotations
from math import gcd, lcm
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE = None
NOTEBOOK_ID = "NB_04_GDT_TES_FABRICATION"
AUDIT_ID = "GDT_AUDIT_01"

SOURCE_FILES = [
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
    "SOURCE_03_electroplating_process.yaml",
    "SOURCE_04_thermal_conductivity.yaml",
]

def find_repo_root():
    start = Path.cwd().resolve()
    candidates = [start, *start.parents, Path("/content/sensors-becker"),
                  Path("/home/dan/sensors-becker"), Path.home()/"sensors-becker"]
    if REPO_ROOT_OVERRIDE:
        candidates.insert(0, Path(REPO_ROOT_OVERRIDE).expanduser().resolve())
    for c in candidates:
        if c.is_dir() and (c/"engineering_navigator").is_dir():
            return c
    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(["git","clone",REPOSITORY_URL,str(target)], check=True)
        if (target/"engineering_navigator").is_dir():
            return target
    raise FileNotFoundError("Could not locate sensors-becker")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OBJ_DIR = ROOT/"engineering_navigator"/"engineering_objects"
SOURCE_DIR = ROOT/"engineering_navigator"/"absorber_manufacturing"/"source_records"
PROCESS_STATUS = ROOT/"outputs"/"engineering_questions"/"absorber_manufacturing"/"PROCESS_WINDOW_01"/"process_window_status.json"
OUTPUT_DIR = ROOT/"outputs"/"engineering_questions"/"absorber_manufacturing"/AUDIT_ID
EXPORT_DIR = ROOT/"exports"/AUDIT_ID
EXPORT_ZIP = ROOT/"exports"/f"{AUDIT_ID}_export.zip"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)


## 2. Load Engineering Objects and source records


In [ ]:
def load_yaml(path):
    data = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected mapping")
    return data

engineering_objects = {
    k: load_yaml(OBJ_DIR/f"{k}.yaml")
    for k in ("absorber","electroplating","tes")
}

records = {}
for fn in SOURCE_FILES:
    r = load_yaml(SOURCE_DIR/fn)
    records[r["source_id"]] = r

if PROCESS_STATUS.exists():
    process_window_status = json.loads(PROCESS_STATUS.read_text(encoding="utf-8"))
else:
    process_window_status = {
        "status":"measurement_plan_ready",
        "numeric_process_window_established":False,
        "source_supported_operating_points":len(
            engineering_objects["electroplating"].get("reported_process_points",[])
        ),
        "leading_process_variables":[
            "Bi_thickness","grain_size","current_density","bias_voltage","plating_rate"
        ],
    }

print("Objects:", ", ".join(engineering_objects))
print("Sources:", ", ".join(sorted(records)))
print("Process-window status:", process_window_status.get("status"))


## 3. General Divisor Theorem computational implementation


In [ ]:
def prime_factors(n):
    if n <= 0:
        raise ValueError("n must be positive")
    out=set(); x=n; p=2
    while p*p <= x:
        while x%p==0:
            out.add(p); x//=p
        p += 1
    if x>1:
        out.add(x)
    return out

def radical(n):
    r=1
    for p in prime_factors(n):
        r*=p
    return r

def phi(n):
    r=n
    for p in prime_factors(n):
        r -= r//p
    return r

def gdt_parameters(N,m,a):
    if N<=0 or m<=0:
        raise ValueError("N,m must be positive")
    d = radical(gcd(m,N))
    R = radical(N)//d
    branch = gcd(a,d)==1
    return {
        "N":N,"m":m,"a":a,"d":d,"R":R,
        "admissible_branch":branch,
        "minimal_period":lcm(m,radical(N)),
        "accepted_per_period":phi(R) if branch else 0,
        "correction_factor":(d/phi(d)) if branch else None,
    }

def gdt_states(N,m,a,L):
    if not gdt_parameters(N,m,a)["admissible_branch"]:
        return []
    return [n for n in range(1,L+1) if n%m==a%m and gcd(n,N)==1]

# Mathematical control only — not a TES application.
print(gdt_parameters(30,6,1))
print(gdt_states(30,6,1,30))


## 4. Collect the current engineering variables


In [ ]:
rows=[]
for oid,obj in engineering_objects.items():
    for item in obj.get("variables",[]):
        if isinstance(item,dict) and item.get("id"):
            rows.append({
                "engineering_object":oid,
                "variable":item.get("id"),
                "unit":item.get("unit",""),
            })
engineering_variables = pd.DataFrame(rows).drop_duplicates()
engineering_variables


## 5. Search current evidence for explicit GDT structure


In [ ]:
MODULAR_TERMS=("modulo","residue","congruence","integer cycle")
COPRIME_TERMS=("coprime","relatively prime","greatest common divisor","gcd")
INTEGER_TERMS=("integer state","discrete state","channel index","pixel index","array index")

def source_text(record):
    fields=("design_variables","reported_values","measured_outcomes",
            "engineering_relationships","engineering_constraints",
            "future_questions","unreported_variables","reported_process_points")
    return " ".join(str(record.get(f,"")) for f in fields).lower()

rows=[]
for sid,r in records.items():
    t=source_text(r)
    ih=[x for x in INTEGER_TERMS if x in t]
    mh=[x for x in MODULAR_TERMS if x in t]
    ch=[x for x in COPRIME_TERMS if x in t]
    rows.append({
        "source_id":sid,
        "integer_state_hits":"; ".join(ih),
        "modular_hits":"; ".join(mh),
        "coprimality_hits":"; ".join(ch),
        "direct_gdt_structure_detected":bool(ih and mh and ch),
    })
source_structure_audit=pd.DataFrame(rows)
source_structure_audit


## 6. Audit each leading fabrication variable


In [ ]:
leading_variables = process_window_status.get(
    "leading_process_variables",
    ["Bi_thickness","grain_size","current_density","bias_voltage","plating_rate"]
)

audit=[]
for variable in leading_variables:
    audit.append({
        "variable":variable,
        "integer_state_or_encoding":"",
        "integer_encoding_physical_basis":"",
        "natural_modulus_m":"",
        "residue_a":"",
        "residue_constraint_physical_basis":"",
        "coprimality_modulus_N":"",
        "coprimality_constraint_physical_basis":"",
        "gdt_hypotheses_satisfied":False,
        "status":"direct_application_not_established",
        "blocker":(
            "Current records provide process/measurement values but no natural "
            "integer residue-class plus coprimality constraint for this variable."
        ),
    })
gdt_applicability_audit=pd.DataFrame(audit)
gdt_applicability_audit


## 7. Physically justified candidate mappings

Add a mapping here only if every arithmetic element has a physical engineering basis. Decimal-to-integer scaling by itself is not a basis.


In [ ]:
GDT_CANDIDATE_MAPPINGS = [
    # {
    #   "candidate_id":"TES_GDT_001",
    #   "engineering_object":"electroplating",
    #   "state_variable":"...",
    #   "N":30, "m":6, "a":1,
    #   "integer_encoding_physical_basis":"...",
    #   "residue_constraint_physical_basis":"...",
    #   "coprimality_constraint_physical_basis":"...",
    #   "evaluation_limit_L":120,
    # }
]
print("Physically justified candidate mappings:", len(GDT_CANDIDATE_MAPPINGS))


## 8. Evaluate justified mappings


In [ ]:
candidate_results=[]
for c in GDT_CANDIDATE_MAPPINGS:
    basis_fields=(
        "integer_encoding_physical_basis",
        "residue_constraint_physical_basis",
        "coprimality_constraint_physical_basis",
    )
    missing=[f for f in basis_fields if not str(c.get(f,"")).strip()]
    if missing:
        candidate_results.append({
            "candidate_id":c.get("candidate_id"),
            "status":"rejected_missing_physical_basis",
            "missing_basis":"; ".join(missing),
        })
        continue
    N,m,a=int(c["N"]),int(c["m"]),int(c["a"])
    L=int(c.get("evaluation_limit_L",0))
    params=gdt_parameters(N,m,a)
    states=gdt_states(N,m,a,L)
    candidate_results.append({
        "candidate_id":c.get("candidate_id"),
        "engineering_object":c.get("engineering_object"),
        "state_variable":c.get("state_variable"),
        "status":"gdt_evaluated",
        **params,
        "evaluation_limit_L":L,
        "accepted_state_count_to_L":len(states),
        "accepted_states_to_L":states,
    })

gdt_candidate_results=pd.DataFrame(candidate_results)
gdt_candidate_results


## 9. Current applicability conclusion


In [ ]:
if not gdt_candidate_results.empty and "status" in gdt_candidate_results:
    evaluated=(gdt_candidate_results["status"]=="gdt_evaluated").any()
    rejected=(gdt_candidate_results["status"]=="rejected_missing_physical_basis").any()
else:
    evaluated=rejected=False

if evaluated:
    overall_status="direct_application_established"
elif rejected:
    overall_status="candidate_mapping_requires_physical_basis"
else:
    overall_status="direct_application_not_established"

print("GDT → TES fabrication:", overall_status)


## 10. Missing specification required for a direct GDT application


In [ ]:
missing_specifications=pd.DataFrame([
    {
        "required_element":"integer engineering state",
        "current_status":"not established",
        "question":"Is there a naturally discrete fabrication/design state?",
        "next_evidence":"Identify a physically discrete state rather than an arbitrary binning.",
    },
    {
        "required_element":"natural modulus m",
        "current_status":"not established",
        "question":"Does a physical fabrication/design rule repeat with an integer period?",
        "next_evidence":"Document a physically defined integer periodicity.",
    },
    {
        "required_element":"residue class a mod m",
        "current_status":"not established",
        "question":"Does a physical rule select one residue class?",
        "next_evidence":"Document the engineering rule selecting the residue.",
    },
    {
        "required_element":"coprimality gcd(n,N)=1",
        "current_status":"not established",
        "question":"Does compatibility/exclusion exactly correspond to avoiding factors of N?",
        "next_evidence":"Identify a real factor-exclusion rule.",
    },
])
missing_specifications


## 11. Becker-facing interpretation

If the result is `direct_application_not_established`, the defensible statement is:

> I tested whether the General Divisor Theorem directly applies to the TES-fabrication constraints represented by the published evidence. The current process variables are measured physical/process quantities, but the records do not yet establish the theorem's required residue-class and coprimality structure. I therefore rejected a direct theorem application rather than discretizing the variables arbitrarily.

If a real discrete/factor-exclusion rule is later identified, this notebook becomes the point where the GDT computes exact admissible states and the next fabrication states to test.


## 12. Write outputs


In [ ]:
source_csv=OUTPUT_DIR/"source_structure_audit.csv"
audit_csv=OUTPUT_DIR/"gdt_applicability_audit.csv"
candidate_csv=OUTPUT_DIR/"gdt_candidate_results.csv"
missing_csv=OUTPUT_DIR/"missing_gdt_specifications.csv"
status_json=OUTPUT_DIR/"gdt_tes_status.json"

source_structure_audit.to_csv(source_csv,index=False)
gdt_applicability_audit.to_csv(audit_csv,index=False)
gdt_candidate_results.to_csv(candidate_csv,index=False)
missing_specifications.to_csv(missing_csv,index=False)

status={
    "notebook_id":NOTEBOOK_ID,
    "status":overall_status,
    "theorem":"General Divisor Theorem",
    "sensor_application":"TES fabrication",
    "source_ids":sorted(records),
    "leading_process_variables":list(leading_variables),
    "physically_justified_candidate_mapping_count":len(GDT_CANDIDATE_MAPPINGS),
    "direct_application_claim_allowed":overall_status=="direct_application_established",
    "current_conclusion":(
        "A direct GDT application to TES fabrication is not yet established by "
        "the current engineering/source constraints."
        if overall_status=="direct_application_not_established"
        else "See evaluated candidate mappings."
    ),
    "next_engineering_step":(
        "Identify a physically natural discrete TES state with an explicit residue "
        "constraint and coprimality constraint, or retain the rejection of direct applicability."
    ),
}
status_json.write_text(json.dumps(status,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")

written={
    "source_structure_audit":source_csv,
    "gdt_applicability_audit":audit_csv,
    "gdt_candidate_results":candidate_csv,
    "missing_gdt_specifications":missing_csv,
    "gdt_tes_status":status_json,
}
for name,path in written.items():
    print(f"{name:28} {path.relative_to(ROOT)}")
status


## 13. Build and download export ZIP


In [ ]:
shutil.rmtree(EXPORT_DIR,ignore_errors=True)
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
for path in written.values():
    shutil.copy2(path,EXPORT_DIR/path.name)
if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()
with zipfile.ZipFile(EXPORT_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            z.write(path,arcname=path.name)
print("Export package:",EXPORT_ZIP)
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 14. Handoff

NB_04 v1.0 is the first explicit bridge between the General Divisor Theorem and a real engineering application candidate.

It earns a positive application claim only where the engineering problem itself supplies the GDT hypotheses.

```text
TES evidence
    ↓
Engineering Objects
    ↓
GDT applicability audit
    ├── direct structure found → exact admissible states → next states to test
    └── structure absent       → missing specification → ask for real discrete/factor rule
```

*Admissible generalizations trail leading specifications.*
